In [ ]:
%%time

import os
import re
import json
import nltk
import string
import random
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import unicodedata
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

# TensorFlow & Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sklearn utilities
from sklearn.utils import class_weight
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, roc_curve, precision_recall_curve
)
from sklearn.model_selection import train_test_split

# NLTK utilities
nltk.download('punkt')
from nltk.tokenize import word_tokenize, ToktokTokenizer
from nltk.corpus import stopwords

# Suppress warnings
warnings.filterwarnings('ignore')

# Enable inline plotting (for Jupyter Notebook)
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


CPU times: user 12.7 s, sys: 1.96 s, total: 14.7 s
Wall time: 27.8 s


# LSTM

In [ ]:
vi_train_df=pd.read_csv('/content/vio-train-balanced.csv')
ag_binary_train_df=pd.read_csv('/content/ag-train-binary-balanced.csv')
em_train_df=pd.read_csv('/content/em-train-balanced.csv')
snt_train_df=pd.read_csv('/content/snt_balanced_train.csv')

nltk.download('stopwords')
nltk.download('punkt')
tokenizer = Tokenizer()
tokenizer.fit_on_texts(pd.concat([vi_train_df['text'], ag_binary_train_df['text'], em_train_df['text'], snt_train_df['text']]))
violence_sequences = tokenizer.texts_to_sequences(vi_train_df['text'])
aggressive_sequences = tokenizer.texts_to_sequences(ag_binary_train_df['text'])
emotion_sequences = tokenizer.texts_to_sequences(em_train_df['text'])
sentement_sequences = tokenizer.texts_to_sequences(snt_train_df['text'])
# Padding

max_length = 100
emotion_padded = pad_sequences(emotion_sequences, maxlen = max_length, padding = 'post')
violence_padded = pad_sequences(violence_sequences, maxlen = max_length, padding = 'post')
aggressive_padded = pad_sequences(aggressive_sequences, maxlen = max_length, padding = 'post')
sentement_padded = pad_sequences(sentement_sequences, maxlen = max_length, padding = 'post')

#generating labels in numpy array format
emotion_labels = np.array(em_train_df['label'])
violence_labels = np.array(vi_train_df['label'])
aggressive_labels = np.array(ag_binary_train_df['label'])
sentement_labels = np.array(snt_train_df['label'])

#prepare seperate inputs for each dataset
emotion_input = emotion_padded
violence_input = violence_padded
aggressive_input = aggressive_padded
sentement_input = sentement_padded

#defining multiple input layers for each task
emotion_input_layer = keras.layers.Input(shape = (max_length,), name = 'emotion_input')
violence_input_layer = keras.layers.Input(shape = (max_length,), name = 'violence_input')
aggressive_input_layer = keras.layers.Input(shape = (max_length,), name = 'aggressive_input')
sentement_input_layer = keras.layers.Input(shape = (max_length,), name = 'sentement_input')

#use as Shared embedding layer
embedding_layer = keras.layers.Embedding(input_dim = len(tokenizer.word_index) + 1, output_dim = 128)
#APPLY THE EMBEDDING LAYER TO EACH INPUT
emotion_embedding = embedding_layer(emotion_input_layer)
violence_embedding = embedding_layer(violence_input_layer)
aggressive_embedding = embedding_layer(aggressive_input_layer)
sentement_embedding = embedding_layer(sentement_input_layer)

#shared LSTM layer
shared_lstm = keras.layers.LSTM(64, return_sequences = True)
emotion_lstm = shared_lstm(emotion_embedding)
violence_lstm = shared_lstm(violence_embedding)
aggressive_lstm = shared_lstm(aggressive_embedding)
sentement_lstm = shared_lstm(sentement_embedding)

#shared global average pooling layer and dropout layer
shared_pooling = keras.layers.GlobalAveragePooling1D()
shared_dropout = keras.layers.Dropout(0.5)
emotion_features = shared_dropout(shared_pooling(emotion_lstm))
violence_features = shared_dropout(shared_pooling(violence_lstm))
aggressive_features = shared_dropout(shared_pooling(aggressive_lstm))
sentement_features = shared_dropout(shared_pooling(sentement_lstm))

#output layers
emotion_output = keras.layers.Dense(6, activation = 'softmax', name = 'emotion_output')(emotion_features)
violence_output = keras.layers.Dense(3, activation = 'softmax', name = 'violence_output')(violence_features)
aggressive_output = keras.layers.Dense(2, activation = 'softmax', name = 'aggressive_output')(aggressive_features)
sentement_output = keras.layers.Dense(2, activation = 'softmax', name = 'sentement_output')(sentement_features)

#compile the model with multiple inputs and outputs
model = keras.models.Model(inputs = [emotion_input_layer, violence_input_layer, aggressive_input_layer, sentement_input_layer],
                           outputs = [emotion_output, violence_output, aggressive_output, sentement_output])

model.compile(optimizer = 'adam',
              loss = {
                  'emotion_output' : 'sparse_categorical_crossentropy',
                  'violence_output' :'sparse_categorical_crossentropy',
                  'aggressive_output' : 'sparse_categorical_crossentropy',
                  'sentement_output' : 'sparse_categorical_crossentropy'
              },
              metrics = {
                  'emotion_output': 'accuracy',
                  'violence_output': 'accuracy',
                  'aggressive_output':'accuracy',
                  'sentement_output':'accuracy'
              })
#training the model with sepearte inputs
model.fit(x = {'emotion_input' : emotion_input,
               'violence_input' : violence_input,
               'aggressive_input' : aggressive_input,
               'sentement_input' : sentement_input},
          y = {'emotion_output' : emotion_labels,
               'violence_output' : violence_labels,
               'aggressive_output' : aggressive_labels,
               'sentement_output' : sentement_labels},
          epochs = 6,
          batch_size = 4)
model.save("/content/lstm_multi_task_model.h5")



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Epoch 1/6
675/675 ━━━━━━━━━━━━━━━━━━━━ 111s 153ms/step - aggressive_output_accuracy: 0.5686 - aggressive_output_loss: 0.6685 - emotion_output_accuracy: 0.1662 - emotion_output_loss: 1.8082 - loss: 4.1901 - sentement_output_accuracy: 0.5045 - sentement_output_loss: 0.6996 - violence_output_accuracy: 0.5109 - violence_output_loss: 1.0138
Epoch 2/6
675/675 ━━━━━━━━━━━━━━━━━━━━ 142s 153ms/step - aggressive_output_accuracy: 0.8964 - aggressive_output_loss: 0.3255 - emotion_output_accuracy: 0.2140 - emotion_output_loss: 1.7801 - loss: 3.7381 - sentement_output_accuracy: 0.5850 - sentement_output_loss: 0.6732 - violence_output_accuracy: 0.5341 - violence_output_loss: 0.9594
Epoch 3/6
675/675 ━━━━━━━━━━━━━━━━━━━━ 143s 155ms/step - aggressive_output_accuracy: 0.9440 - aggressive_output_loss: 0.1678 - emotion_output_accuracy: 0.2810 - emotion_output_loss: 1.6649 - loss: 3.0970 - sentement_output_accuracy: 0.7289 - sentement_output_loss: 0.5774 - violence_output_accuracy: 0.7325 - violence_output

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report

# Load validation datasets
vi_val_df = pd.read_csv("/content/vio-dev-balanced.csv")
ag_val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
em_val_df = pd.read_csv("/content/em-val-balanced.csv")
snt_val_df = pd.read_csv("/content/snt_balanced_dev.csv")

# Use the same tokenizer from training
violence_sequences_val = tokenizer.texts_to_sequences(vi_val_df['text'])
aggressive_sequences_val = tokenizer.texts_to_sequences(ag_val_df['text'])
emotion_sequences_val = tokenizer.texts_to_sequences(em_val_df['text'])
sentement_sequences_val = tokenizer.texts_to_sequences(snt_val_df['text'])

# Padding (same max_length used in training)
max_length = 100
emotion_padded_val = pad_sequences(emotion_sequences_val, maxlen=max_length, padding='post')
violence_padded_val = pad_sequences(violence_sequences_val, maxlen=max_length, padding='post')
aggressive_padded_val = pad_sequences(aggressive_sequences_val, maxlen=max_length, padding='post')
sentement_padded_val = pad_sequences(sentement_sequences_val, maxlen=max_length, padding='post')

# Convert labels to numpy arrays
emotion_labels_val = np.array(em_val_df['label'])
violence_labels_val = np.array(vi_val_df['label'])
aggressive_labels_val = np.array(ag_val_df['label'])
sentement_labels_val = np.array(snt_val_df['label'])

#prepare seperate inputs for each dataset
emotion_input = emotion_padded_val
violence_input = violence_padded_val
aggressive_input = aggressive_padded_val
sentement_input = sentement_padded_val

import numpy as np
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report

# Load the trained model
model = load_model("/content/lstm_multi_task_model.h5")  # Change path if needed

# Make predictions
predictions = model.predict({'emotion_input': emotion_input,
                             'violence_input': violence_input,
                             'aggressive_input': aggressive_input,
                             'sentement_input': sentement_input})

# Extract outputs for each task (list indices instead of dictionary keys)
emotion_preds = np.argmax(predictions[0], axis=1)  # First output
violence_preds = np.argmax(predictions[1], axis=1)  # Second output
aggressive_preds = np.argmax(predictions[2], axis=1)  # Third output
sentement_preds = np.argmax(predictions[3], axis=1)  # Fourth output

# Evaluate model performance
print("\nSentiment Classification Report:")
print(classification_report(emotion_labels_val, emotion_preds))

print("\nViolence Classification Report:")
print(classification_report(violence_labels_val, violence_preds))

print("\nAggression Classification Report:")
print(classification_report(aggressive_labels_val, aggressive_preds))

print("\nSentement Classification Report:")
print(classification_report(sentement_labels_val, sentement_preds))




20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step

Sentiment Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.42      0.49       120
           1       0.48      0.47      0.47       129
           2       0.31      0.42      0.36        64
           3       0.54      0.54      0.54       155
           4       0.33      0.42      0.37        67
           5       0.57      0.54      0.55        89

    accuracy                           0.48       624
   macro avg       0.47      0.47      0.46       624
weighted avg       0.49      0.48      0.48       624


Violence Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.76      0.61       214
           1       0.52      0.54      0.53       214
           2       0.81      0.34      0.48       196

    accuracy                           0.55       624
   macro avg       0.61      0.55      0.54       624
weighted avg       0.61 

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report

# Load validation datasets
vi_test_df = pd.read_csv("/content/vio-test-balanced.csv")
ag_test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")
em_test_df = pd.read_csv("/content/em-test-balanced.csv")
snt_test_df = pd.read_csv("/content/snt_balanced_test.csv")

# Use the same tokenizer from training
violence_sequences_test = tokenizer.texts_to_sequences(vi_test_df['text'])
aggressive_sequences_test = tokenizer.texts_to_sequences(ag_test_df['text'])
emotion_sequences_test = tokenizer.texts_to_sequences(em_test_df['text'])
sentement_sequences_test = tokenizer.texts_to_sequences(snt_test_df['text'])

# Padding (same max_length used in training)
max_length = 100
emotion_padded_test = pad_sequences(emotion_sequences_test, maxlen=max_length, padding='post')
violence_padded_test = pad_sequences(violence_sequences_test, maxlen=max_length, padding='post')
aggressive_padded_test = pad_sequences(aggressive_sequences_test, maxlen=max_length, padding='post')
sentement_padded_test = pad_sequences(sentement_sequences_test, maxlen=max_length, padding='post')

# Convert labels to numpy arrays
emotion_labels_test = np.array(em_test_df['label'])
violence_labels_test = np.array(vi_test_df['label'])
aggressive_labels_test = np.array(ag_test_df['label'])
sentement_labels_test = np.array(snt_test_df['label'])

#prepare seperate inputs for each dataset
emotion_input = emotion_padded_test
violence_input = violence_padded_test
aggressive_input = aggressive_padded_test
sentement_input = sentement_padded_test

import numpy as np
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report

# Load the trained model
model = load_model("/content/lstm_multi_task_model.h5")  # Change path if needed

# Make predictions
predictions = model.predict({'emotion_input': emotion_input,
                             'violence_input': violence_input,
                             'aggressive_input': aggressive_input,
                             'sentement_input' : sentement_input})

# Extract outputs for each task (list indices instead of dictionary keys)
emotion_preds = np.argmax(predictions[0], axis=1)  # First output
violence_preds = np.argmax(predictions[1], axis=1)  # Second output
aggressive_preds = np.argmax(predictions[2], axis=1)  # Third output
sentement_preds = np.argmax(predictions[3], axis=1)  # Fourth output

# Evaluate model performance
print("\nSentiment Classification Report:")
print(classification_report(emotion_labels_test, emotion_preds))

print("\nViolence Classification Report:")
print(classification_report(violence_labels_test, violence_preds))

print("\nAggression Classification Report:")
print(classification_report(aggressive_labels_test, aggressive_preds))

print("\nSentement Classification Report:")
print(classification_report(sentement_labels_test, sentement_preds))


20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 91ms/step

Sentiment Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.43      0.47       114
           1       0.46      0.34      0.39       119
           2       0.34      0.42      0.38        73
           3       0.50      0.49      0.50       165
           4       0.27      0.38      0.32        71
           5       0.49      0.53      0.51        83

    accuracy                           0.44       625
   macro avg       0.43      0.43      0.43       625
weighted avg       0.45      0.44      0.44       625


Violence Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.80      0.62       212
           1       0.53      0.51      0.52       212
           2       0.80      0.36      0.49       201

    accuracy                           0.56       625
   macro avg       0.61      0.55      0.55       625
weighted avg       0.61 

# BILSTM

In [ ]:
import pandas as pd
import numpy as np
import nltk
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow import keras

# Load datasets
vi_train_df = pd.read_csv('/content/vio-train-balanced.csv')
ag_binary_train_df = pd.read_csv('/content/ag-train-binary-balanced.csv')
em_train_df = pd.read_csv('/content/em-train-balanced.csv')
snt_train_df = pd.read_csv('/content/snt_balanced_train.csv')

# Download stopwords and punkt tokenizer
nltk.download('stopwords')
nltk.download('punkt')

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(pd.concat([vi_train_df['text'], ag_binary_train_df['text'], em_train_df['text'], snt_train_df['text'] ]))

violence_sequences = tokenizer.texts_to_sequences(vi_train_df['text'])
aggressive_sequences = tokenizer.texts_to_sequences(ag_binary_train_df['text'])
emotion_sequences = tokenizer.texts_to_sequences(em_train_df['text'])
sentiment_sequences = tokenizer.texts_to_sequences(snt_train_df['text'])

# Padding
max_length = 100
emotion_padded = pad_sequences(emotion_sequences, maxlen=max_length, padding='post')
violence_padded = pad_sequences(violence_sequences, maxlen=max_length, padding='post')
aggressive_padded = pad_sequences(aggressive_sequences, maxlen=max_length, padding='post')
sentiment_padded = pad_sequences(sentiment_sequences, maxlen=max_length, padding='post')

# Labels
emotion_labels = np.array(em_train_df['label'])
violence_labels = np.array(vi_train_df['label'])
aggressive_labels = np.array(ag_binary_train_df['label'])
sentiment_labels = np.array(snt_train_df['label'])

# Inputs
emotion_input = emotion_padded
violence_input = violence_padded
aggressive_input = aggressive_padded
sentiment_input = sentiment_padded

# Input layers
emotion_input_layer = keras.layers.Input(shape=(max_length,), name='emotion_input')
violence_input_layer = keras.layers.Input(shape=(max_length,), name='violence_input')
aggressive_input_layer = keras.layers.Input(shape=(max_length,), name='aggressive_input')
sentiment_input_layer = keras.layers.Input(shape=(max_length,), name='sentiment_input')

# Shared embedding layer
embedding_layer = keras.layers.Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=128)

# Apply embedding
emotion_embedding = embedding_layer(emotion_input_layer)
violence_embedding = embedding_layer(violence_input_layer)
aggressive_embedding = embedding_layer(aggressive_input_layer)
sentiment_embedding = embedding_layer(sentiment_input_layer)

# Shared BiLSTM layer (replaces LSTM with Bidirectional LSTM)
shared_bilstm = keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True))

emotion_bilstm = shared_bilstm(emotion_embedding)
violence_bilstm = shared_bilstm(violence_embedding)
aggressive_bilstm = shared_bilstm(aggressive_embedding)
sentiment_bilstm = shared_bilstm(sentiment_embedding)

# Shared pooling and dropout
shared_pooling = keras.layers.GlobalAveragePooling1D()
shared_dropout = keras.layers.Dropout(0.5)

emotion_features = shared_dropout(shared_pooling(emotion_bilstm))
violence_features = shared_dropout(shared_pooling(violence_bilstm))
aggressive_features = shared_dropout(shared_pooling(aggressive_bilstm))
sentiment_features = shared_dropout(shared_pooling(sentiment_bilstm))

# Output layers
emotion_output = keras.layers.Dense(6, activation='softmax', name='emotion_output')(emotion_features)
violence_output = keras.layers.Dense(3, activation='softmax', name='violence_output')(violence_features)
aggressive_output = keras.layers.Dense(2, activation='softmax', name='aggressive_output')(aggressive_features)
sentiment_output = keras.layers.Dense(2, activation='softmax', name='sentiment_output')(sentiment_features)

# Compile model
model = keras.models.Model(
    inputs=[emotion_input_layer, violence_input_layer, aggressive_input_layer, sentiment_input_layer],
    outputs=[emotion_output, violence_output, aggressive_output, sentiment_output]
)

model.compile(
    optimizer='adam',
    loss={
        'emotion_output': 'sparse_categorical_crossentropy',
        'violence_output': 'sparse_categorical_crossentropy',
        'aggressive_output': 'sparse_categorical_crossentropy',
        'sentiment_output': 'sparse_categorical_crossentropy'
    },
    metrics={
        'emotion_output': 'accuracy',
        'violence_output': 'accuracy',
        'aggressive_output': 'accuracy',
        'sentiment_output': 'accuracy'
    }
)

# Train model
model.fit(
    x={
        'emotion_input': emotion_input,
        'violence_input': violence_input,
        'aggressive_input': aggressive_input,
        'sentiment_input': sentiment_input
    },
    y={
        'emotion_output': emotion_labels,
        'violence_output': violence_labels,
        'aggressive_output': aggressive_labels,
        'sentiment_output': sentiment_labels
    },
    epochs=5,
    batch_size=4
)

# Save model
model.save("/content/bilstm_multi_task_model.h5")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Epoch 1/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 209s 294ms/step - aggressive_output_accuracy: 0.5548 - aggressive_output_loss: 0.6763 - emotion_output_accuracy: 0.1746 - emotion_output_loss: 1.8063 - loss: 4.1850 - sentiment_output_accuracy: 0.5141 - sentiment_output_loss: 0.6935 - violence_output_accuracy: 0.5099 - violence_output_loss: 1.0089
Epoch 2/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 188s 278ms/step - aggressive_output_accuracy: 0.8456 - aggressive_output_loss: 0.3894 - emotion_output_accuracy: 0.2254 - emotion_output_loss: 1.7765 - loss: 3.8255 - sentiment_output_accuracy: 0.6077 - sentiment_output_loss: 0.6715 - violence_output_accuracy: 0.5130 - violence_output_loss: 0.9880
Epoch 3/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 202s 279ms/step - aggressive_output_accuracy: 0.9288 - aggressive_output_loss: 0.1876 - emotion_output_accuracy: 0.3316 - emotion_output_loss: 1.6162 - loss: 3.0870 - sentiment_output_accuracy: 0.7958 - sentiment_output_loss: 0.4844 - violence_output_accuracy: 0.6534 - violence_output

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report
import pickle

# Load validation datasets
vi_val_df = pd.read_csv("/content/vio-dev-balanced.csv")
ag_val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
em_val_df = pd.read_csv("/content/em-val-balanced.csv")
snt_val_df = pd.read_csv("/content/snt_balanced_dev.csv")

# Tokenization

violence_sequences_val = tokenizer.texts_to_sequences(vi_val_df['text'])
aggressive_sequences_val = tokenizer.texts_to_sequences(ag_val_df['text'])
emotion_sequences_val = tokenizer.texts_to_sequences(em_val_df['text'])
sentiment_sequences_val = tokenizer.texts_to_sequences(snt_val_df['text'])

# Padding (same as used during training)
max_length = 100
emotion_input = pad_sequences(emotion_sequences_val, maxlen=max_length, padding='post')
violence_input = pad_sequences(violence_sequences_val, maxlen=max_length, padding='post')
aggressive_input = pad_sequences(aggressive_sequences_val, maxlen=max_length, padding='post')
sentiment_input = pad_sequences(sentiment_sequences_val, maxlen=max_length, padding='post')

# Prepare labels
emotion_labels_val = np.array(em_val_df['label'])
violence_labels_val = np.array(vi_val_df['label'])
aggressive_labels_val = np.array(ag_val_df['label'])
sentiment_labels_val = np.array(snt_val_df['label'])

# Load the trained BiLSTM model
model = load_model("/content/bilstm_multi_task_model.h5")

# Predict on validation data
predictions = model.predict({
    'emotion_input': emotion_input,
    'violence_input': violence_input,
    'aggressive_input': aggressive_input,
    'sentiment_input': sentiment_input
})


# Get predicted labels
emotion_preds = np.argmax(predictions[0], axis=1)
violence_preds = np.argmax(predictions[1], axis=1)
aggressive_preds = np.argmax(predictions[2], axis=1)
sentiment_preds = np.argmax(predictions[3], axis=1)

# Classification Reports
print("\n Emotion Classification Report:")
print(classification_report(emotion_labels_val, emotion_preds))

print("\n Violence Classification Report:")
print(classification_report(violence_labels_val, violence_preds))

print("\n Aggression Classification Report:")
print(classification_report(aggressive_labels_val, aggressive_preds))

print("\n Sentement Classification Report:")
print(classification_report(sentiment_labels_val, sentiment_preds))


20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 171ms/step

 Emotion Classification Report:
              precision    recall  f1-score   support

           0       0.42      0.37      0.39       120
           1       0.43      0.30      0.35       129
           2       0.29      0.47      0.36        64
           3       0.45      0.37      0.41       155
           4       0.28      0.36      0.31        67
           5       0.47      0.58      0.52        89

    accuracy                           0.39       624
   macro avg       0.39      0.41      0.39       624
weighted avg       0.41      0.39      0.39       624


 Violence Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.80      0.61       214
           1       0.47      0.48      0.47       214
           2       0.61      0.17      0.27       196

    accuracy                           0.49       624
   macro avg       0.52      0.48      0.45       624
weighted avg       0.52

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report

# Load test datasets
vi_test_df = pd.read_csv("/content/vio-test-balanced.csv")
ag_test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")
em_test_df = pd.read_csv("/content/em-test-balanced.csv")
snt_test_df = pd.read_csv("/content/snt_balanced_test.csv")


# Tokenize test texts
violence_sequences_test = tokenizer.texts_to_sequences(vi_test_df['text'])
aggressive_sequences_test = tokenizer.texts_to_sequences(ag_test_df['text'])
emotion_sequences_test = tokenizer.texts_to_sequences(em_test_df['text'])
sentiment_sequences_test = tokenizer.texts_to_sequences(snt_test_df['text'])

# Padding (must match the max_length from training)
max_length = 100
emotion_input = pad_sequences(emotion_sequences_test, maxlen=max_length, padding='post')
violence_input = pad_sequences(violence_sequences_test, maxlen=max_length, padding='post')
aggressive_input = pad_sequences(aggressive_sequences_test, maxlen=max_length, padding='post')
sentiment_input = pad_sequences(sentiment_sequences_test, maxlen=max_length, padding='post')

# Convert true labels to numpy arrays
emotion_labels_test = np.array(em_test_df['label'])
violence_labels_test = np.array(vi_test_df['label'])
aggressive_labels_test = np.array(ag_test_df['label'])
sentiment_labels_test = np.array(snt_test_df['label'])

# Load trained BiLSTM model
model = load_model("/content/bilstm_multi_task_model.h5")

# Predict on test data
predictions = model.predict({
    'emotion_input': emotion_input,
    'violence_input': violence_input,
    'aggressive_input': aggressive_input,
    'sentiment_input': sentiment_input
})

# Get predicted class labels
emotion_preds = np.argmax(predictions[0], axis=1)
violence_preds = np.argmax(predictions[1], axis=1)
aggressive_preds = np.argmax(predictions[2], axis=1)
sentiment_preds = np.argmax(predictions[3], axis=1)

# Evaluate each task
print("\n🔹 Sentiment Classification Report:")
print(classification_report(emotion_labels_test, emotion_preds))

print("\n🔹 Violence Classification Report:")
print(classification_report(violence_labels_test, violence_preds))

print("\n🔹 Aggression Classification Report:")
print(classification_report(aggressive_labels_test, aggressive_preds))

print("\n🔹 Sentement Classification Report:")
print(classification_report(sentiment_labels_test, sentiment_preds))


20/20 ━━━━━━━━━━━━━━━━━━━━ 14s 200ms/step

🔹 Sentiment Classification Report:
              precision    recall  f1-score   support

           0       0.58      0.47      0.52       114
           1       0.40      0.28      0.33       119
           2       0.26      0.41      0.32        73
           3       0.55      0.39      0.46       165
           4       0.32      0.46      0.38        71
           5       0.29      0.40      0.34        83

    accuracy                           0.40       625
   macro avg       0.40      0.40      0.39       625
weighted avg       0.43      0.40      0.40       625


🔹 Violence Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.81      0.62       212
           1       0.44      0.48      0.46       212
           2       0.62      0.17      0.27       201

    accuracy                           0.49       625
   macro avg       0.52      0.48      0.45       625
weighted avg      

# LSTM+BILSTM

In [ ]:
import pandas as pd
import numpy as np
import nltk
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow import keras

# Load datasets
vi_train_df = pd.read_csv('/content/vio-train-balanced.csv')
ag_binary_train_df = pd.read_csv('/content/ag-train-binary-balanced.csv')
em_train_df = pd.read_csv('/content/em-train-balanced.csv')
snt_train_df = pd.read_csv('/content/snt_balanced_train.csv')

# Download NLTK resources
nltk.download('stopwords')
nltk.download('punkt')

# Tokenizer (fit on all text data combined)
tokenizer = Tokenizer()
tokenizer.fit_on_texts(pd.concat([vi_train_df['text'], ag_binary_train_df['text'], em_train_df['text'], snt_train_df['text']]))

# Convert texts to sequences
violence_sequences = tokenizer.texts_to_sequences(vi_train_df['text'])
aggressive_sequences = tokenizer.texts_to_sequences(ag_binary_train_df['text'])
emotion_sequences = tokenizer.texts_to_sequences(em_train_df['text'])
sentiment_sequences = tokenizer.texts_to_sequences(snt_train_df['text'])

# Padding
max_length = 100
emotion_input = pad_sequences(emotion_sequences, maxlen=max_length, padding='post')
violence_input = pad_sequences(violence_sequences, maxlen=max_length, padding='post')
aggressive_input = pad_sequences(aggressive_sequences, maxlen=max_length, padding='post')
sentiment_input = pad_sequences(sentiment_sequences, maxlen=max_length, padding='post')

# Labels
emotion_labels = np.array(em_train_df['label'])
violence_labels = np.array(vi_train_df['label'])
aggressive_labels = np.array(ag_binary_train_df['label'])
sentiment_labels = np.array(snt_train_df['label'])

# Input Layers
emotion_input_layer = keras.layers.Input(shape=(max_length,), name='emotion_input')
violence_input_layer = keras.layers.Input(shape=(max_length,), name='violence_input')
aggressive_input_layer = keras.layers.Input(shape=(max_length,), name='aggressive_input')
sentiment_input_layer = keras.layers.Input(shape=(max_length,), name='sentiment_input')

# Shared Embedding
embedding_layer = keras.layers.Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=128)

# Define function to apply LSTM + BiLSTM
def lstm_bilstm_branch(input_layer):
    x = embedding_layer(input_layer)
    x = keras.layers.LSTM(64, return_sequences=True)(x)
    x = keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True))(x)
    x = keras.layers.GlobalAveragePooling1D()(x)
    x = keras.layers.Dropout(0.5)(x)
    return x

# Branches for each task
emotion_features = lstm_bilstm_branch(emotion_input_layer)
violence_features = lstm_bilstm_branch(violence_input_layer)
aggressive_features = lstm_bilstm_branch(aggressive_input_layer)
sentiment_features = lstm_bilstm_branch(sentiment_input_layer)

# Output Layers
emotion_output = keras.layers.Dense(6, activation='softmax', name='emotion_output')(emotion_features)
violence_output = keras.layers.Dense(3, activation='softmax', name='violence_output')(violence_features)
aggressive_output = keras.layers.Dense(2, activation='softmax', name='aggressive_output')(aggressive_features)
sentiment_output = keras.layers.Dense(2, activation='softmax', name='sentiment_output')(sentiment_features)

# Build and Compile Model
model = keras.models.Model(
    inputs=[emotion_input_layer, violence_input_layer, aggressive_input_layer, sentiment_input_layer],
    outputs=[emotion_output, violence_output, aggressive_output, sentiment_output]
)

model.compile(
    optimizer='adam',
    loss={
        'emotion_output': 'sparse_categorical_crossentropy',
        'violence_output': 'sparse_categorical_crossentropy',
        'aggressive_output': 'sparse_categorical_crossentropy',
        'sentiment_output': 'sparse_categorical_crossentropy'
    },
    metrics={
        'emotion_output': 'accuracy',
        'violence_output': 'accuracy',
        'aggressive_output': 'accuracy',
        'sentiment_output': 'accuracy'
    }
)

# Train the Model
model.fit(
    x={
        'emotion_input': emotion_input,
        'violence_input': violence_input,
        'aggressive_input': aggressive_input,
        'sentiment_input': sentiment_input
    },
    y={
        'emotion_output': emotion_labels,
        'violence_output': violence_labels,
        'aggressive_output': aggressive_labels,
        'sentiment_output': sentiment_labels
    },
    epochs=15,
    batch_size=4
)

# Save the Trained Model
model.save("/content/lstm+bilstm_multi_task_model.h5")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Epoch 1/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 293s 402ms/step - aggressive_output_accuracy: 0.5591 - aggressive_output_loss: 0.6661 - emotion_output_accuracy: 0.1659 - emotion_output_loss: 1.8002 - loss: 4.1783 - sentiment_output_accuracy: 0.4887 - sentiment_output_loss: 0.7013 - violence_output_accuracy: 0.5009 - violence_output_loss: 1.0107
Epoch 2/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 402s 521ms/step - aggressive_output_accuracy: 0.9206 - aggressive_output_loss: 0.2649 - emotion_output_accuracy: 0.2302 - emotion_output_loss: 1.7442 - loss: 3.6196 - sentiment_output_accuracy: 0.4969 - sentiment_output_loss: 0.7013 - violence_output_accuracy: 0.5732 - violence_output_loss: 0.9093
Epoch 3/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 282s 372ms/step - aggressive_output_accuracy: 0.9731 - aggressive_output_loss: 0.1012 - emotion_output_accuracy: 0.3363 - emotion_output_loss: 1.4190 - loss: 2.9349 - sentiment_output_accuracy: 0.6185 - sentiment_output_loss: 0.6455 - violence_output_accuracy: 0.6701 - violence_out

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report
import pickle

# Load validation datasets
vi_val_df = pd.read_csv("/content/vio-dev-balanced.csv")
ag_val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
em_val_df = pd.read_csv("/content/em-val-balanced.csv")
snt_val_df = pd.read_csv("/content/snt_balanced_dev.csv")

# Tokenize text data
violence_sequences_val = tokenizer.texts_to_sequences(vi_val_df['text'])
aggressive_sequences_val = tokenizer.texts_to_sequences(ag_val_df['text'])
emotion_sequences_val = tokenizer.texts_to_sequences(em_val_df['text'])
sentiment_sequences_val = tokenizer.texts_to_sequences(snt_val_df['text'])

# Pad sequences to fixed max length (same as used in training)
max_length = 100
emotion_input = pad_sequences(emotion_sequences_val, maxlen=max_length, padding='post')
violence_input = pad_sequences(violence_sequences_val, maxlen=max_length, padding='post')
aggressive_input = pad_sequences(aggressive_sequences_val, maxlen=max_length, padding='post')
sentiment_input = pad_sequences(sentiment_sequences_val, maxlen=max_length, padding='post')

# Convert true labels to numpy arrays
emotion_labels_val = np.array(em_val_df['label'])
violence_labels_val = np.array(vi_val_df['label'])
aggressive_labels_val = np.array(ag_val_df['label'])
sentiment_labels_val = np.array(snt_val_df['label'])

# Load the trained LSTM + BiLSTM multi-task model
model = load_model("/content/lstm+bilstm_multi_task_model.h5")

# Predict on validation data
predictions = model.predict({
    'emotion_input': emotion_input,
    'violence_input': violence_input,
    'aggressive_input': aggressive_input,
    'sentiment_input': sentiment_input
})

# Convert probabilities to predicted class indices
emotion_preds = np.argmax(predictions[0], axis=1)
violence_preds = np.argmax(predictions[1], axis=1)
aggressive_preds = np.argmax(predictions[2], axis=1)
sentiment_preds = np.argmax(predictions[3], axis=1)

# Print classification reports
print("\n🔹 Emotion Classification Report:")
print(classification_report(emotion_labels_val, emotion_preds))

print("\n🔹 Violence Classification Report:")
print(classification_report(violence_labels_val, violence_preds))

print("\n🔹 Aggression Classification Report:")
print(classification_report(aggressive_labels_val, aggressive_preds))

print("\n🔹 Sentiment Classification Report:")
print(classification_report(sentiment_labels_val, sentiment_preds))


20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 240ms/step

🔹 Emotion Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.41      0.49       120
           1       0.42      0.40      0.41       129
           2       0.43      0.55      0.48        64
           3       0.41      0.68      0.51       155
           4       0.50      0.25      0.34        67
           5       0.63      0.33      0.43        89

    accuracy                           0.46       624
   macro avg       0.50      0.44      0.44       624
weighted avg       0.49      0.46      0.45       624


🔹 Violence Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.77      0.65       214
           1       0.56      0.57      0.57       214
           2       0.72      0.42      0.53       196

    accuracy                           0.59       624
   macro avg       0.61      0.58      0.58       624
weighted avg       0.

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report
import pickle


# Load test datasets
vi_test_df = pd.read_csv("/content/vio-test-balanced.csv")
ag_test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")
em_test_df = pd.read_csv("/content/em-test-balanced.csv")
snt_test_df = pd.read_csv("/content/snt_balanced_test.csv")

# Convert texts to sequences
violence_sequences_test = tokenizer.texts_to_sequences(vi_test_df['text'])
aggressive_sequences_test = tokenizer.texts_to_sequences(ag_test_df['text'])
emotion_sequences_test = tokenizer.texts_to_sequences(em_test_df['text'])
sentiment_sequences_test = tokenizer.texts_to_sequences(snt_test_df['text'])

# Pad sequences to the same max length as used during training
max_length = 100
emotion_input = pad_sequences(emotion_sequences_test, maxlen=max_length, padding='post')
violence_input = pad_sequences(violence_sequences_test, maxlen=max_length, padding='post')
aggressive_input = pad_sequences(aggressive_sequences_test, maxlen=max_length, padding='post')
sentiment_input = pad_sequences(sentiment_sequences_test, maxlen=max_length, padding='post')

# Extract true labels
emotion_labels_test = np.array(em_test_df['label'])
violence_labels_test = np.array(vi_test_df['label'])
aggressive_labels_test = np.array(ag_test_df['label'])
sentiment_labels_test = np.array(snt_test_df['label'])

# Load the trained LSTM+BiLSTM model
model = load_model("/content/lstm+bilstm_multi_task_model.h5")

# Predict on test inputs
predictions = model.predict({
    'emotion_input': emotion_input,
    'violence_input': violence_input,
    'aggressive_input': aggressive_input,
    'sentiment_input': sentiment_input
})

# Get predicted class indices
emotion_preds = np.argmax(predictions[0], axis=1)
violence_preds = np.argmax(predictions[1], axis=1)
aggressive_preds = np.argmax(predictions[2], axis=1)
sentiment_preds = np.argmax(predictions[3], axis=1)

# Print classification reports
print("\n🔹 Sentiment Classification Report:")
print(classification_report(emotion_labels_test, emotion_preds))

print("\n🔹 Violence Classification Report:")
print(classification_report(violence_labels_test, violence_preds))

print("\n🔹 Aggression Classification Report:")
print(classification_report(aggressive_labels_test, aggressive_preds))

print("\n🔹 Sentement Classification Report:")
print(classification_report(sentiment_labels_test, sentiment_preds))


20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 228ms/step

🔹 Sentiment Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.48      0.58       114
           1       0.50      0.45      0.47       119
           2       0.42      0.49      0.46        73
           3       0.43      0.73      0.54       165
           4       0.47      0.24      0.32        71
           5       0.73      0.40      0.52        83

    accuracy                           0.50       625
   macro avg       0.55      0.46      0.48       625
weighted avg       0.54      0.50      0.50       625


🔹 Violence Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.74      0.63       212
           1       0.55      0.57      0.56       212
           2       0.71      0.44      0.55       201

    accuracy                           0.58       625
   macro avg       0.61      0.58      0.58       625
weighted avg       

# LSTM+BILSTM+CNN

In [ ]:
import pandas as pd
import numpy as np
import nltk
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow import keras

# Load datasets
vi_train_df = pd.read_csv('/content/vio-train-balanced.csv')
ag_binary_train_df = pd.read_csv('/content/ag-train-binary-balanced.csv')
em_train_df = pd.read_csv('/content/em-train-balanced.csv')
snt_train_df = pd.read_csv('/content/snt_balanced_train.csv')

# Download NLTK resources
nltk.download('stopwords')
nltk.download('punkt')

# Tokenizer (fit on all text data combined)
tokenizer = Tokenizer()
tokenizer.fit_on_texts(pd.concat([vi_train_df['text'], ag_binary_train_df['text'], em_train_df['text'], snt_train_df['text']]))

# Convert texts to sequences
violence_sequences = tokenizer.texts_to_sequences(vi_train_df['text'])
aggressive_sequences = tokenizer.texts_to_sequences(ag_binary_train_df['text'])
emotion_sequences = tokenizer.texts_to_sequences(em_train_df['text'])
sentiment_sequences = tokenizer.texts_to_sequences(snt_train_df['text'])

# Padding
max_length = 100
emotion_input = pad_sequences(emotion_sequences, maxlen=max_length, padding='post')
violence_input = pad_sequences(violence_sequences, maxlen=max_length, padding='post')
aggressive_input = pad_sequences(aggressive_sequences, maxlen=max_length, padding='post')
sentiment_input = pad_sequences(sentiment_sequences, maxlen=max_length, padding='post')

# Labels
emotion_labels = np.array(em_train_df['label'])
violence_labels = np.array(vi_train_df['label'])
aggressive_labels = np.array(ag_binary_train_df['label'])
sentiment_labels = np.array(snt_train_df['label'])

# Input Layers
emotion_input_layer = keras.layers.Input(shape=(max_length,), name='emotion_input')
violence_input_layer = keras.layers.Input(shape=(max_length,), name='violence_input')
aggressive_input_layer = keras.layers.Input(shape=(max_length,), name='aggressive_input')
sentiment_input_layer = keras.layers.Input(shape=(max_length,), name='sentiment_input')

# Shared Embedding
embedding_layer = keras.layers.Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=128)

# Define function to apply LSTM + BiLSTM + CNN
def lstm_bilstm_cnn_branch(input_layer):
    x = embedding_layer(input_layer)
    x = keras.layers.LSTM(64, return_sequences=True)(x)
    x = keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True))(x)
    x = keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling1D(pool_size=2)(x)
    x = keras.layers.GlobalAveragePooling1D()(x)
    x = keras.layers.Dropout(0.5)(x)
    return x

# Branches for each task
emotion_features = lstm_bilstm_cnn_branch(emotion_input_layer)
violence_features = lstm_bilstm_cnn_branch(violence_input_layer)
aggressive_features = lstm_bilstm_cnn_branch(aggressive_input_layer)
sentiment_features = lstm_bilstm_cnn_branch(sentiment_input_layer)

# Output Layers
emotion_output = keras.layers.Dense(6, activation='softmax', name='emotion_output')(emotion_features)
violence_output = keras.layers.Dense(3, activation='softmax', name='violence_output')(violence_features)
aggressive_output = keras.layers.Dense(2, activation='softmax', name='aggressive_output')(aggressive_features)
sentiment_output = keras.layers.Dense(2, activation='softmax', name='sentiment_output')(sentiment_features)

# Build and Compile Model
model = keras.models.Model(
    inputs=[emotion_input_layer, violence_input_layer, aggressive_input_layer, sentiment_input_layer],
    outputs=[emotion_output, violence_output, aggressive_output, sentiment_output]
)

model.compile(
    optimizer='adam',
    loss={
        'emotion_output': 'sparse_categorical_crossentropy',
        'violence_output': 'sparse_categorical_crossentropy',
        'aggressive_output': 'sparse_categorical_crossentropy',
        'sentiment_output': 'sparse_categorical_crossentropy'
    },
    metrics={
        'emotion_output': 'accuracy',
        'violence_output': 'accuracy',
        'aggressive_output': 'accuracy',
        'sentiment_output': 'accuracy'
    }
)

# Train the Model
model.fit(
    x={
        'emotion_input': emotion_input,
        'violence_input': violence_input,
        'aggressive_input': aggressive_input,
        'sentiment_input': sentiment_input
    },
    y={
        'emotion_output': emotion_labels,
        'violence_output': violence_labels,
        'aggressive_output': aggressive_labels,
        'sentiment_output': sentiment_labels
    },
    epochs=10,
    batch_size=4
)

# Save the Trained Model
model.save("/content/lstm+bilstm+cnn_multi_task_model.h5")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Epoch 1/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 317s 434ms/step - aggressive_output_accuracy: 0.6225 - aggressive_output_loss: 0.6349 - emotion_output_accuracy: 0.1702 - emotion_output_loss: 1.8001 - loss: 4.1560 - sentiment_output_accuracy: 0.4834 - sentiment_output_loss: 0.6994 - violence_output_accuracy: 0.4966 - violence_output_loss: 1.0215
Epoch 2/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 322s 434ms/step - aggressive_output_accuracy: 0.9339 - aggressive_output_loss: 0.1995 - emotion_output_accuracy: 0.2428 - emotion_output_loss: 1.6966 - loss: 3.5006 - sentiment_output_accuracy: 0.5486 - sentiment_output_loss: 0.6884 - violence_output_accuracy: 0.5757 - violence_output_loss: 0.9160
Epoch 3/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 323s 435ms/step - aggressive_output_accuracy: 0.9807 - aggressive_output_loss: 0.0824 - emotion_output_accuracy: 0.4088 - emotion_output_loss: 1.2813 - loss: 2.2451 - sentiment_output_accuracy: 0.8930 - sentiment_output_loss: 0.3635 - violence_output_accuracy: 0.8006 - violence_out

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report

# Load validation datasets
vi_val_df = pd.read_csv("/content/vio-dev-balanced.csv")
ag_val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
em_val_df = pd.read_csv("/content/em-val-balanced.csv")
snt_val_df = pd.read_csv("/content/snt_balanced_dev.csv")



# Convert text to sequences
violence_sequences_val = tokenizer.texts_to_sequences(vi_val_df['text'])
aggressive_sequences_val = tokenizer.texts_to_sequences(ag_val_df['text'])
emotion_sequences_val = tokenizer.texts_to_sequences(em_val_df['text'])
sentiment_sequences_val = tokenizer.texts_to_sequences(snt_val_df['text'])

# Pad sequences
max_length = 100
emotion_input = pad_sequences(emotion_sequences_val, maxlen=max_length, padding='post')
violence_input = pad_sequences(violence_sequences_val, maxlen=max_length, padding='post')
aggressive_input = pad_sequences(aggressive_sequences_val, maxlen=max_length, padding='post')
sentiment_input = pad_sequences(sentiment_sequences_val, maxlen=max_length, padding='post')

# Extract true labels

# True labels
emotion_labels_val = np.array(em_val_df['label'])
violence_labels_val = np.array(vi_val_df['label'])
aggressive_labels_val = np.array(ag_val_df['label'])
sentiment_labels_val = np.array(snt_val_df['label'])

# Load the trained LSTM+BiLSTM+CNN model
model = load_model("/content/lstm+bilstm+cnn_multi_task_model.h5")

# Predict on validation data
predictions = model.predict({
    'emotion_input': emotion_input,
    'violence_input': violence_input,
    'aggressive_input': aggressive_input,
    'sentiment_input': sentiment_input
})

# Get predicted labels
emotion_preds = np.argmax(predictions[0], axis=1)
violence_preds = np.argmax(predictions[1], axis=1)
aggressive_preds = np.argmax(predictions[2], axis=1)
sentiment_preds = np.argmax(predictions[3], axis=1)

# Classification Reports
print("\n🔹 Sentiment Classification Report:")
print(classification_report(emotion_labels_val, emotion_preds))

print("\n🔹 Violence Classification Report:")
print(classification_report(violence_labels_val, violence_preds))

print("\n🔹 Aggression Classification Report:")
print(classification_report(aggressive_labels_val, aggressive_preds))

print("\n🔹 Sentiment Classification Report:")
print(classification_report(sentiment_labels_val, sentiment_preds))

20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 267ms/step

🔹 Sentiment Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.43      0.49       120
           1       0.46      0.40      0.43       129
           2       0.43      0.52      0.47        64
           3       0.58      0.58      0.58       155
           4       0.40      0.52      0.45        67
           5       0.53      0.61      0.57        89

    accuracy                           0.50       624
   macro avg       0.49      0.51      0.50       624
weighted avg       0.51      0.50      0.50       624


🔹 Violence Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.82      0.63       214
           1       0.65      0.48      0.55       214
           2       0.79      0.49      0.61       196

    accuracy                           0.60       624
   macro avg       0.65      0.60      0.60       624
weighted avg       

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report
import pickle

# Load test datasets
vi_test_df = pd.read_csv("/content/vio-test-balanced.csv")
ag_test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")
em_test_df = pd.read_csv("/content/em-test-balanced.csv")
snt_test_df = pd.read_csv("/content/snt_balanced_test.csv")

# Convert texts to sequences
violence_sequences_test = tokenizer.texts_to_sequences(vi_test_df['text'])
aggressive_sequences_test = tokenizer.texts_to_sequences(ag_test_df['text'])
emotion_sequences_test = tokenizer.texts_to_sequences(em_test_df['text'])
sentiment_sequences_test = tokenizer.texts_to_sequences(snt_test_df['text'])

# Pad sequences using the same max length as training
max_length = 100
emotion_input = pad_sequences(emotion_sequences_test, maxlen=max_length, padding='post')
violence_input = pad_sequences(violence_sequences_test, maxlen=max_length, padding='post')
aggressive_input = pad_sequences(aggressive_sequences_test, maxlen=max_length, padding='post')
sentiment_input = pad_sequences(sentiment_sequences_test, maxlen=max_length, padding='post')

# Convert labels to numpy arrays
emotion_labels_test = np.array(em_test_df['label'])
violence_labels_test = np.array(vi_test_df['label'])
aggressive_labels_test = np.array(ag_test_df['label'])
sentiment_labels_test = np.array(snt_test_df['label'])

# ✅ Load the trained LSTM+BiLSTM+CNN model
model = load_model("/content/lstm+bilstm+cnn_multi_task_model.h5")

# Predict on test inputs
predictions = model.predict({
    'emotion_input': emotion_input,
    'violence_input': violence_input,
    'aggressive_input': aggressive_input,
    'sentiment_input': sentiment_input
})

# Convert probabilities to class indices
emotion_preds = np.argmax(predictions[0], axis=1)
violence_preds = np.argmax(predictions[1], axis=1)
aggressive_preds = np.argmax(predictions[2], axis=1)
sentiment_preds = np.argmax(predictions[3], axis=1)

# Print classification reports
print("\n🔹 Sentiment Classification Report:")
print(classification_report(emotion_labels_test, emotion_preds))

print("\n🔹 Violence Classification Report:")
print(classification_report(violence_labels_test, violence_preds))

print("\n🔹 Aggression Classification Report:")
print(classification_report(aggressive_labels_test, aggressive_preds))

print("\n🔹 Sentiment Classification Report:")
print(classification_report(sentiment_labels_test, sentiment_preds))


20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 252ms/step

🔹 Sentiment Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.49      0.54       114
           1       0.42      0.37      0.39       119
           2       0.42      0.47      0.44        73
           3       0.51      0.52      0.52       165
           4       0.35      0.45      0.39        71
           5       0.55      0.58      0.56        83

    accuracy                           0.48       625
   macro avg       0.48      0.48      0.48       625
weighted avg       0.49      0.48      0.48       625


🔹 Violence Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.85      0.64       212
           1       0.62      0.36      0.46       212
           2       0.75      0.55      0.64       201

    accuracy                           0.59       625
   macro avg       0.63      0.59      0.58       625
weighted avg       